# fase 1b do Churn Intelligence â€” scraping e NLP das reclamaÃ§Ãµes da Claro

nesse notebook, vamo pra segunda etapa da fase 1 do projeto. enquanto a fase 1a trabalhou com o dataset estruturado do Kaggle, aqui a gente coleta e processa dados nÃ£o estruturados reais, com as reclamaÃ§Ãµes pÃºblicas da Claro no Reclame Aqui.

o objetivo aqui Ã© transformar o texto bruto de clientes insatisfeitos em features de sentimento que vÃ£o alimentar o pipeline de RAG na fase 3.

antes de comeÃ§ar a desenvolver o cÃ³digo em si, entrei na pÃ¡gina da Claro no Reclame Aqui, analisei o DevTools e filtrei por Fetch/XHR na aba Network. na lista das requisiÃ§Ãµes que estavam sendo feitas, procurei pelas requisiÃ§Ãµes que retornavam um JSON com os dados das reclamaÃ§Ãµes pra ver a URL completa, os headers enviados e a resposta JSON com os dados estruturados das reclamaÃ§Ãµes.

a URL encontrada foi: https://iosearch.reclameaqui.com.br/raichu-io-site-search-v1/query/companyComplains/5/10?company=7712

e a lista completa dos headers foi:
- Accept: application/json, text/plain, */*,
- Accept-Encoding: gzip, deflate, br, zstd,
- Accept-Language: en-US,en;q=0.9,pt;q=0.8,
- Origin: https://www.reclameaqui.com.br,
- Referer: https://www.reclameaqui.com.br/,
- Sec-Ch-Ua: 'Chromium;v=146, Not-A.Brand;v=24, Google Chrome;v=146',
- Sec-Ch-Ua-Mobile: ?0,
- Sec-Ch-Ua-Platform: 'Windows',
- Sec-Fetch-Dest: empty,
- Sec-Fetch-Mode: cors,
- Sec-Fetch-Site: same-site,
- User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36

com essas informaÃ§Ãµes, a gente jÃ¡ consegue comeÃ§ar a scrapear as reclamaÃ§Ãµes

In [ ]:
# vamo tentar usar request simples e ver se funciona

import requests
import pandas as pd
import time

# vamo usar o "User-Agent" aqi pro site nÃ£o bloquear por saber que Ã© um bot
headers = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Accept-Language": "en-US,en;q=0.9,pt;q=0.8",
    "Origin": "https://www.reclameaqui.com.br",
    "Referer": "https://www.reclameaqui.com.br/",
    "Sec-Ch-Ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"
}

reclamacoes = []

for offset in range(0, 500, 5):  #-> pra iterar de 0 a 500 pulando de 5 em 5
    url = f"https://iosearch.reclameaqui.com.br/raichu-io-site-search-v1/query/companyComplains/5/{offset}?company=7712"

    # pra fazer a requisiÃ§Ã£o HTTP pra API, passando os headers que imitam um navegador
    response = requests.get(url, headers=headers)

    if response.status_code == 200: #-> 200 Ã© "sucesso"
        data = response.json()
        # pra navegar pelo JSON atÃ© chegar na lista de reclamaÃ§Ãµes
        complains = data['complainResult']['complains']['data']
        reclamacoes.extend(complains)
        print(f"offset {offset} â€” {len(complains)} reclamaÃ§Ãµes coletadas")
    else:
        print(f"offset {offset} â€” erro {response.status_code}")

    # vamo usar um sleep de 1s entre requisiÃ§Ãµes pra nÃ£o sobrecarregar o servidor e evitar bloqueio
    time.sleep(1)

df_reclamacoes = pd.DataFrame(reclamacoes)
print(df_reclamacoes.shape)
print(df_reclamacoes.columns.tolist())

offset 0 â€” erro 403
offset 5 â€” erro 403
offset 10 â€” erro 403
offset 15 â€” erro 403
offset 20 â€” erro 403
offset 25 â€” erro 403
offset 30 â€” erro 403
offset 35 â€” erro 403
offset 40 â€” erro 403
offset 45 â€” erro 403
offset 50 â€” erro 403
offset 55 â€” erro 403
offset 60 â€” erro 403
offset 65 â€” erro 403
offset 70 â€” erro 403
offset 75 â€” erro 403
offset 80 â€” erro 403
offset 85 â€” erro 403
offset 90 â€” erro 403
offset 95 â€” erro 403
offset 100 â€” erro 403
offset 105 â€” erro 403
offset 110 â€” erro 403
offset 115 â€” erro 403
offset 120 â€” erro 403
offset 125 â€” erro 403
offset 130 â€” erro 403
offset 135 â€” erro 403
offset 140 â€” erro 403
offset 145 â€” erro 403
offset 150 â€” erro 403
offset 155 â€” erro 403
offset 160 â€” erro 403
offset 165 â€” erro 403
offset 170 â€” erro 403
offset 175 â€” erro 403
offset 180 â€” erro 403
offset 185 â€” erro 403
offset 190 â€” erro 403
offset 195 â€” erro 403
offset 200 â€” erro 403
offset 205 â€” erro 403
offset 210 â€”

o request chama a API certa e recebe o JSON estruturado, mas o Reclame Aqui usa Cloudflare, que bloqueia a requisiÃ§Ã£o.

Quando um navegador real acessa o site, o Cloudflare executa JavaScript invisÃ­vel que verifica dezenas de sinais, como movimento do mouse, fingerprint do navegador, tempo de carregamento, etc. sÃ³ depois disso ele emite o cookie cf_clearance. o requests Ã© uma biblioteca HTTP pura, o que significa que ele nÃ£o executa JavaScript, entÃ£o nunca consegue gerar esse cookie por conta prÃ³pria.

entÃ£o, em vez  de imitar um navegador, vamo usar Playwright que resolve isso controlando um navegador Chromium real, executando o JavaScript de verificaÃ§Ã£o do Cloudflare normalmente e gerando o cookie vÃ¡lido. depois que o cookie existe na sessÃ£o do navegador, a gente faz o fetch da API de dentro do contexto do navegador, entÃ£o o Cloudflare nÃ£o consegue distinguir de um usuÃ¡rio real.

o Colab roda em servidores do Google, cujos IPs sÃ£o imediatamente reconhecidos e bloqueados pelo Cloudflare. alÃ©m disso, o Playwright precisa instalar o browser Chromium, que encontra alguns probelmas aqui no Colab. entÃ£o vamo rodar o scraper localmente e depois retornar com o csv pra seguir aqui.

In [18]:
# vamo carregar o csv gerado pelo scraping com as reclamaÃ§Ãµes da claro no reclame aqui:

import pandas as pd

url = "https://raw.githubusercontent.com/meurii/churn-intelligence/refs/heads/main/data/reclamacoes_claro.csv"
df_reclamacoes = pd.read_csv(url)
print(df_reclamacoes.shape)
df_reclamacoes.head()

(254, 69)


,lastReplyOrigin,oldComplainId,additionalFields,moderationUserName,companyName,otherProblemType,solved,dealAgain,otherProductType,titleMasked,...,userName,url,inModeration,moderateRequested,deleted,complainMediaInfos,fantasyName,category,descriptionMasked,user
0,NaN,NaN,NaN,****,Claro,NaN,False,NaN,ServiÃ§o TÃ©cnico Residencial,ReclamaÃ§Ã£o sobre Atendimento TÃ©cnico: ServiÃ§o ...,...,****,reclamacao-sobre-atendimento-tecnico-servico-m...,False,NaN,False,[],Claro,255.0,Assunto: ReclamaÃ§Ã£o Atendimento TÃ©cnico<br/><b...,NaN
1,NaN,NaN,NaN,****,Claro,NaN,False,NaN,NaN,CobranÃ§a indevida de multa por portabilidade a...,...,****,cobranca-indevida-de-multa-por-portabilidade-a...,False,NaN,False,[],Claro,67.0,Fiz uma portabilidade a uns 4 anos atrÃ¡s para ...,NaN
2,NaN,NaN,NaN,****,Claro,NaN,False,NaN,NaN,Claro: AssÃ©dio telefÃ´nico com ligaÃ§Ãµes excessi...,...,****,claro-assedio-telefonico-com-ligacoes-excessiv...,False,NaN,False,[],Claro,66.0,Gostaria de registrar minha indignaÃ§Ã£o com a q...,NaN
3,NaN,NaN,NaN,****,Claro,NaN,False,NaN,NaN,CobranÃ§a duplicada e nÃ£o estornada pela Claro ...,...,****,cobranca-duplicada-e-nao-estornada-pela-claro-...,False,NaN,False,[],Claro,67.0,Em dezembro de 2025 paguei a fatura de novembr...,NaN
4,NaN,NaN,NaN,****,Claro,NaN,False,NaN,NaN,Internet Empresarial com Reparo TÃ©cnico Penden...,...,****,internet-empresarial-com-reparo-tecnico-penden...,False,NaN,False,[],Claro,69.0,JÃ¡ solicitei reparo tÃ©cnico para a internet du...,NaN


apesar da Claro ter mais de 400 mil reclamaÃ§Ãµes na plataforma, a gente sÃ³ conseguiu registrar 254 no processo de scrapping, chegando a um limite de offset em torno de 255. Ã© bem provÃ¡vel que essa limitaÃ§Ã£o seja intencional, jÃ¡ que o Reclame Aqui oferece acesso completo aos dados via planos pagos da sua API comercial. como a intenÃ§Ã£o aqui Ã© fazer uma exploraÃ§Ã£o das possibilidades do modelo, a gente usou o acesso pÃºblico via API interna que, provavelmente, restringe o acesso justamente pra proteger esse modelo de negÃ³cio.

tentativas de aumentar o tamanho da pÃ¡gina via parÃ¢metro na URL resultaram
em bloqueio de IP pelo Cloudflare. o limite de offset em 255 tbm se mostrou
consistente em mÃºltiplas execuÃ§Ãµes independentes, confirmando que a API
pÃºblica simplesmente nÃ£o entrega dados alÃ©m desse ponto pra requisiÃ§Ãµes
nÃ£o autenticadas.

as reclamaÃ§Ãµes que a gente conseguiu scrapear sÃ£o informaÃ§Ãµes reais, com texto completo, data, categoria e status de resoluÃ§Ã£o. entÃ£o, vamo considerar que isso Ã© suficiente pra alimentar o pipeline de NLP e o sistema RAG da fase 3.

o foco do projeto continua sendo:

coleta â†’ sentimento â†’ modelo â†’ explicabilidade â†’ recomendaÃ§Ã£o

e nÃ£o apenas o volume alto e bruto de dados.

In [19]:
# vamo analisar os tipos e nulos das colunas

df_reclamacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 69 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   lastReplyOrigin              0 non-null      float64
 1   oldComplainId                0 non-null      float64
 2   additionalFields             0 non-null      float64
 3   moderationUserName           254 non-null    object 
 4   companyName                  254 non-null    object 
 5   otherProblemType             27 non-null     object 
 6   solved                       254 non-null    bool   
 7   dealAgain                    0 non-null      float64
 8   otherProductType             60 non-null     object 
 9   titleMasked                  254 non-null    object 
 10  type                         0 non-null      float64
 11  evaluation                   0 non-null      float64
 12  contentPoliciesViolation     3 non-null      object 
 13  score               

In [20]:
# vamo eliminar todas as colunas que tÃ£o completamente vazias

df_reclamacoes = df_reclamacoes.dropna(axis=1, how='all')
print(df_reclamacoes.shape)
df_reclamacoes.info() #-> pra gente ver as que sobraram

(254, 47)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 47 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   moderationUserName           254 non-null    object 
 1   companyName                  254 non-null    object 
 2   otherProblemType             27 non-null     object 
 3   solved                       254 non-null    bool   
 4   otherProductType             60 non-null     object 
 5   titleMasked                  254 non-null    object 
 6   contentPoliciesViolation     3 non-null      object 
 7   userState                    254 non-null    object 
 8   userRequestedDelete          254 non-null    bool   
 9   modified                     254 non-null    object 
 10  id                           254 non-null    object 
 11  evaluated                    254 non-null    bool   
 12  presence                     186 non-null    object 
 13  deletionRe

In [21]:
# vamo confirmar pelo id se tem alguma linha duplicada

df_reclamacoes.duplicated('id').sum()

np.int64(12)

In [22]:
# vamo remover as 12 linhas duplicatas que a gente encontrou

df_reclamacoes = df_reclamacoes.drop_duplicates(subset='id')
print(df_reclamacoes.shape)  # a gente espera que aqui tenham 242 linhas

(242, 47)


a gente ainda tem 47 colunas, mas nem todas sÃ£o relevantes pro modelo.
- userEmail, userName, ip, deletedIp e address contÃ©m informaÃ§Ãµes pessoais identificÃ¡veis de usuÃ¡rios. entÃ£o, por questÃ£o de privacidade, vamo remover essas colunas;
- moderationUserName, companyName, companyShortname, fantasyName, titleMasked, descriptionMasked, maskingStatus, inModeration, category, moderateReason, moderationReasonDescription, frozen, deleted, deletionReason, userRequestedDelete, productType, otherProblemType, otherProductType, compliment, modified, read, complainMediaInfos, companyIndexes, legacyId, url, complainOrigin, failedToValidatePolicies, contentPoliciesViolation, company, contentViolatesPolicies e policiesViolationScore sÃ£o, basicamente, dados operacionais internos da plataforma que nÃ£o dizem nada sobre a experiÃªncia do cliente, entÃ£o, vamo remover essas colunas tbm;
- por Ãºltimo, id era essencial sÃ³ pra gente confirmar a deduplicaÃ§Ã£o. como jÃ¡ fizemos essa etapa, dÃ¡ pra remover ela tbm.

sobraram as colunas title, description, created, status, solved, hasReply, problemType, userCity, userState, evaluated, presence, interactions. vamo avaliar essas mais a fundo pra decidir quais seguem e quais devem ser removidas, pra manter apenas o que alimenta o NLP, o RAG e o que pode virar feature pro modelo.

In [23]:
# vamo confirmar a variabilidade dessas colunas, pra garantir que elas podem ser Ãºteis pro modelo

print(f"Taxa de resposta da Claro: {df_reclamacoes['solved'].mean():.1%}")
print(f"Taxa de resposta da Claro: {df_reclamacoes['hasReply'].mean():.1%}")
print(f"Taxa de resposta da Claro: {df_reclamacoes['evaluated'].mean():.1%}")

Taxa de resposta da Claro: 0.0%
Taxa de resposta da Claro: 0.0%
Taxa de resposta da Claro: 0.0%


essa anÃ¡lise confirmou o que motivou a escolha da Claro como empresa do estudo:
taxa de resposta de 0% e taxa de resoluÃ§Ã£o de 0% nas 242 reclamaÃ§Ãµes coletadas.
esses dados, embora nÃ£o entrem no modelo por falta de variabilidade, reforÃ§am o
argumento de que um sistema de churn intelligence faria diferenÃ§a real
aqui precisamente porque a empresa nÃ£o tem nenhum processo de resposta ativo.

In [24]:
# vamo avaliar tbm os valores dessas colunas

print(df_reclamacoes['presence'].value_counts())
print(df_reclamacoes['interactions'].value_counts())
print(df_reclamacoes['status'].value_counts())
print(df_reclamacoes['created'].value_counts())
print(df_reclamacoes['problemType'].value_counts())

presence
SERVICE_OFF    176
STORE_ON         1
SERVICE_ON       1
Name: count, dtype: int64
interactions
[]    242
Name: count, dtype: int64
status
PENDING    242
Name: count, dtype: int64
created
2026-04-06T09:15:57    2
2026-04-06T11:18:08    1
2026-04-06T11:17:26    1
2026-04-06T11:17:20    1
2026-04-06T11:20:20    1
                      ..
2026-04-05T16:35:51    1
2026-04-05T16:35:43    1
2026-04-05T16:31:19    1
2026-04-05T16:27:27    1
2026-04-05T16:13:13    1
Name: count, Length: 241, dtype: int64
problemType
0000000000000075                     79
0000000000000875                     13
0000000000000877                     11
0000000000000011                      9
0000000000000868                      8
0000000000000017                      8
0000000000001227                      8
0000000000000078                      5
0000000000000878                      5
0000000000000007                      4
0000000000000045                      4
0000000000000639                     

alÃ©m do evaluated, presence, interactions e status tbm apresentaram valor Ãºnico em 100% dos registros. como jÃ¡ comentei, essa falta de variabilidade nÃ£o agrega nada ao modelo. entÃ£o elas tbm podem ser removidas. por Ãºltimo, problemType retorna cÃ³digos numÃ©ricos que, provavelmente, tÃªm uma tabela de mapeamento interna do Reclame Aqui que nÃ£o Ã© pÃºblica. Sem acesso a essa tabela, os cÃ³digos nÃ£o vÃ£o ter significado utilizÃ¡vel aqui.

concluindo: restam apenas as colunas title, description, created, userCity e userState. essas 5 colunas concentram toda a informaÃ§Ã£o necessÃ¡ria pra gente:
- description e title alimentam o NLP e o RAG;
- created permite anÃ¡lise temporal;
- e userCity e userState dÃ£o contexto geogrÃ¡fico.

In [25]:
# vamo atualizar o df sÃ³ com as colunas que vÃ£o ser relevantes

colunas_relevantes = ['title', 'description', 'created', 'userCity', 'userState']

df_final = df_reclamacoes[colunas_relevantes].copy()
print(df_final.shape)
# e aproveitar pra confirmar se tem algum dado faltando
print(df_final.isnull().sum())

(242, 5)
title          0
description    0
created        0
userCity       0
userState      0
dtype: int64


In [26]:
# vamo precisar fazer uma limpeza de texto nas colunas que vamo usar pra NLP e RAG
# pra isso, vamo comeÃ§ar instalando a biblioteca necessÃ¡ria

!pip install unidecode

In [27]:
# depois criar a funÃ§Ã£o de limpeza

import re
from unidecode import unidecode

def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""

    # pra remover tags HTML
    texto = re.sub(r'<[^>]+>', ' ', texto)

    # pra remover emojis mantendo acentos
    texto = re.sub(r'[^\w\s\.,!?Ã Ã¡Ã¢Ã£Ã¤Ã©ÃªÃ«Ã­Ã®Ã¯Ã³Ã´ÃµÃ¶ÃºÃ»Ã¼Ã§Ã±]', ' ', texto)

    # pra converter para minÃºsculas
    texto = texto.lower()

    # pra remover URLs
    texto = re.sub(r'http\S+|www\S+', '', texto)

    # pra remover nÃºmeros de telefone e CPF (padrÃµes comuns em reclamaÃ§Ãµes)
    texto = re.sub(r'\d{2,}[\.\-]?\d{4,}[\.\-]?\d{4}', '', texto)

    # pra remover pontuaÃ§Ã£o excessiva mantendo . e , que ajudam o modelo
    texto = re.sub(r'[!?]{2,}', '.', texto)  #-> !!! vira .
    texto = re.sub(r'[^\w\s\.,]', ' ', texto)

    # pra remover espaÃ§os extras
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

In [28]:
# e, por ultimo, vamo aplicar nas duas colunas de texto

df_final['description_clean'] = df_final['description'].apply(limpar_texto)
df_final['title_clean'] = df_final['title'].apply(limpar_texto)

# compara original vs limpo em alguns exemplos
for i in range(3):
    print("ORIGINAL:", df_final['description'].iloc[i][:200])
    print("LIMPO:", df_final['description_clean'].iloc[i][:200])
    print("---")

ORIGINAL: Assunto: ReclamaÃ§Ã£o Atendimento TÃ©cnico<br/><br/>Prezados,<br/><br/>Venho formalizar reclamaÃ§Ã£o sobre dois atendimentos tÃ©cnicos r
LIMPO: assunto reclamaÃ§Ã£o atendimento tÃ©cnico prezados, venho formalizar reclamaÃ§Ã£o sobre dois atendimentos tÃ©cnicos r
---
ORIGINAL: Fiz uma portabilidade a uns 4 anos atrÃ¡s para a Tim, sendo que fiquei mais de 2 nos com a claro e estÃ£o me cobrando multa, isso es
LIMPO: fiz uma portabilidade a uns 4 anos atrÃ¡s para a tim, sendo que fiquei mais de 2 nos com a claro e estÃ£o me cobrando multa, isso es
---
ORIGINAL: Gostaria de registrar minha indignaÃ§Ã£o com a quantidade absurda de ligaÃ§Ãµes que eu tenho recebido da Claro. Todos os dias eu sou i
LIMPO: gostaria de registrar minha indignaÃ§Ã£o com a quantidade absurda de ligaÃ§Ãµes que eu tenho recebido da claro. todos os dias eu sou i
---


agora que a gente limpou o texto completamente, vamo passar pra etapa da NLP.
o BERTimbau Ã© o modelo canÃ´nico pra sentimento em textos em portuguÃªs por ser o mais robusto pra produÃ§Ã£o, mas ele tem dois problemas prÃ¡ticos: ele baixa algo prÃ³ximo de 500 MB e Ã© lento em CPU. aqui a gente usa o Colab gratuito, sem GPU, entÃ£o rodar em 242 textos (que Ã© o nosso caso aqui), pode demorar bastante, alÃ©m da sessÃ£o poder cair no meio.

entÃ£o, vamo usar o pysentimiento, que foi treinado especificamente em textos em portuguÃªs do Brasil, incluindo textos de redes sociais e avaliaÃ§Ãµes (que Ã© um perfil similar ao das reclamaÃ§Ãµes do Reclame Aqui). ele Ã© leve, rÃ¡pido em CPU, fÃ¡cil de instalar, e retorna polaridade (positivo, negativo, neutro) com score de confianÃ§a. vai rodar em segundos nos nossos 242 textos.



In [29]:
# vamo passar pra anÃ¡lise de sentimento nas reclamaÃ§Ãµes
# pra isso, vamo comeÃ§ar instalando a biblioteca necessÃ¡ria

!pip install pysentimiento

In [30]:
# vamo ver a distribuiÃ§Ã£o de NEG, NEU e POS nas 242 reclamaÃ§Ãµes

from pysentimiento import create_analyzer
analyzer = create_analyzer(task="sentiment", lang="pt")

df_final['sentimento'] = df_final['description_clean'].apply(
    lambda x: analyzer.predict(x[:512]).output
)

print(df_final['sentimento'].value_counts())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

sentimento
NEU    163
NEG     79
Name: count, dtype: int64


a anÃ¡lise retornou 163 neutros, 79 negativos e nenhum positivo (o que jÃ¡ era esperado, jÃ¡ que sÃ£o reclamaÃ§Ãµes, entÃ£o positivo seria improvÃ¡vel). mas a quantidade de neutros Ã© um pouco surpreendente, jÃ¡ que a gente esperava reclamaÃ§Ãµes majoritariamente negativas. o que pode estar acontecendo aqui Ã© que muitas reclamaÃ§Ãµes podem estar sendo escritas em tom formal e descritivo, narrando o problema sem linguagem emocional explÃ­cita, e por isso, confundindo o modelo, que por ter sido treinado em textos de redes sociais (onde o sentimento costuma ser explÃ­cito), tem dificuldade com esse estilo mais burocrÃ¡tico.

vamo avaliar alguns exemplos de resultados neutros, pra ver se eles fazem sentido ou se sÃ£o falsos neutros que deveriam ser negativos.

In [31]:
# vamo ver exemplos de reclamaÃ§Ãµes classificadas como neutro
neutros = df_final[df_final['sentimento'] == 'NEU']['description_clean']
for texto in neutros[:10]:
    print(texto[:300])
    print("---")

assunto reclamaÃ§Ã£o atendimento tÃ©cnico prezados, venho formalizar reclamaÃ§Ã£o sobre dois atendimentos tÃ©cnicos r
---
em dezembro de 2025 paguei a fatura de novembro 2025 de maneira duplicada. acabei me confundindo pois a claro, por algum motivo, p
---
jÃ¡ solicitei reparo tÃ©cnico para a internet duas vezes, mas ninguÃ©m resolveu ainda. o protocolo do Ãºltimo atendimento Ã© .
---
bom dia em primeiro de dezembro deste ano fiz um contrato com a clara na pessoa do senhor yan no valor de 8490 de dezembro atÃ© hoj
---
tentei fazer um plano mais em conta para minha empresa na claro e os atendentes nÃ£o tinham nenhum plano que fosse mais em conta, e
---
estou recebendo cobranÃ§as de um pagamento que eu jÃ¡ fiz.
---
contratei 5 assinaturas de wi fi mesh em 2023 por emergÃªncia, o sistema parou de funcionar, devolvi os 5 aparelhos e pedi cancelam
---
contratei um linha claro fixo em 25 02 26 com o serviÃ§o de portabilidade de vivo para claro.depois de 22 chamados abertos apÃ³s a c
---
text

o que dÃ¡ pra perceber com alguns desses exemplos Ã© que, realmente, sÃ£o neutros mais pela forma do que pelo conteÃºdo. mesmo com textos descritivos e formais ("venho formalizar reclamaÃ§Ã£o", "bom dia", "contratei", "pedi um chip"), algumas situaÃ§Ãµes sÃ£o graves, como 22 chamados abertos, pagamento duplicado, aparelhos devolvidos sem resoluÃ§Ã£o. as reclamaÃ§Ãµes sÃ£o escritas de forma tÃ£o burocrÃ¡tica que o modelo nÃ£o detecta negatividade, jÃ¡ que ele nÃ£o classifica pelo conteÃºdo semÃ¢ntico da situaÃ§Ã£o.

entÃ£o, aqui a gente encontra uma limitaÃ§Ã£o real do pysentimiento pra esse tipo de texto especÃ­fico. mas de toda forma, mesmo que a feature de sentimento acabe nÃ£o discriminando bem individualmente e nÃ£o entre diretamente no modelo como feature preditiva forte, a distribuiÃ§Ã£o NEG/NEU como um todo ainda Ã© informaÃ§Ã£o Ãºtil pro contexto narrativo no pipeline de RAG.

o entregÃ¡vel dessa fase Ã© o 'reclamacoes_com_sentimento.csv', com o texto limpo de 242 reclamaÃ§Ãµes reais da Claro, e com a classificaÃ§Ã£o de sentimento de cada reclamaÃ§Ã£o.

na fase 3, esse dataset vai alimentar o sistema de RAG, buscando semanticamente nas reclamaÃ§Ãµes os contextos mais relevantes para gerar a recomendaÃ§Ã£o de aÃ§Ã£o quando o modelo identificar o motivo principal do churn via SHAP.

In [32]:
# vamo remover as colunas de description e title com os textos originais
# tbm vamo renomear pra que as limpas nÃ£o precisem ficar com _clean

df_final = df_final.drop(columns=['description', 'title'])
df_final = df_final.rename(columns={
    'description_clean': 'description',
    'title_clean': 'title'
})

print(df_final.columns.tolist())
print(f"shape: {df_final.shape}")
df_final.head()

['created', 'userCity', 'userState', 'description', 'title', 'sentimento']
shape: (242, 6)


,created,userCity,userState,description,title,sentimento
0,2026-04-06T11:18:09,SÃ£o Paulo,SP,assunto reclamaÃ§Ã£o atendimento tÃ©cnico prezado...,reclamaÃ§Ã£o sobre atendimento tÃ©cnico serviÃ§o m...,NEU
1,2026-04-06T11:18:08,FlorianÃ³polis,SC,fiz uma portabilidade a uns 4 anos atrÃ¡s para ...,cobranÃ§a indevida de multa por portabilidade a...,NEG
2,2026-04-06T11:17:26,Nova IguaÃ§u,RJ,gostaria de registrar minha indignaÃ§Ã£o com a q...,claro assÃ©dio telefÃ´nico com ligaÃ§Ãµes excessiv...,NEG
3,2026-04-06T11:17:20,SÃ£o Paulo,SP,em dezembro de 2025 paguei a fatura de novembr...,cobranÃ§a duplicada e nÃ£o estornada pela claro ...,NEU
4,2026-04-06T11:20:20,Rio de Janeiro,RJ,jÃ¡ solicitei reparo tÃ©cnico para a internet du...,internet empresarial com reparo tÃ©cnico penden...,NEU


In [42]:
# por Ãºltimo, vamo salvar o df gerado aqui localmente

from google.colab import files
files.download("reclamacoes_com_sentimento.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>